# Deploy NPS Agent to RHOAI

This notebook deploys the NPS Agent ([`npsagent.py`](./npsagent.py)) to OpenShift AI with MLflow tracing.

## Overview

This notebook takes the agent from the [Evaluate notebook](../1_develop/2_evaluate.ipynb) and deploys it as an HTTP endpoint on **Red Hat OpenShift AI (RHOAI)**. The agent logic is unchanged — we just wrap it in an MLflow `ResponsesAgent` for serving.


### Cluster Setup

Before continuing, you'll need an OpenShift cluster with RHOAI and MLflow already configured. Follow the [Cluster Setup Guide](https://docs.google.com/document/d/1ZzuGAY1gSamOLsznwbaL7xFkJ1JjHWpnyjk0tV12YVg/edit?tab=t.0#heading=h.jgt5ddlrwyvc) to get your environment ready.

---

## Deployment Files Overview

The `2_deploy/` directory is self-contained — everything OpenShift needs to build and run the agent:

| File | What it does |
|---|---|
| [`npsagent.py`](./npsagent.py) | The agent logic (identical to the evaluate notebook) wrapped in an MLflow `ResponsesAgent` so it can serve HTTP requests via `POST /invocations` |
| [`nps_mcp_server.py`](./nps_mcp_server.py) | FastMCP server exposing NPS API tools. Spawned on-demand by `run_nps_agent` via `uv run fastmcp run` — not a long-running process |
| [`app.sh`](./app.sh) | Container entry point. Packages `npsagent.py` with `mlflow.pyfunc.save_model`, then starts `mlflow models serve` on port 8080 (5 min timeout) |
| [`requirements.txt`](./requirements.txt) | Python dependencies installed during the s2i build |
| [`nps-agent.yaml`](./nps-agent.yaml) | OpenShift Template that creates a **BuildConfig** (s2i from `deploydemo` branch using `python:3.12-ubi9`), **ImageStream**, **Deployment** (with secret injection + readiness/liveness probes on `/ping`), **Service** (port 8080), and **Route** (HTTPS with 5 min HAProxy timeout) — all in one `oc process` call |
| [`.s2i/environment`](./.s2i/environment) | Single line: `APP_SCRIPT=app.sh` — tells the [s2i](https://github.com/openshift/source-to-image) builder to use `app.sh` instead of the default Python entrypoint |

---

## What's Different in the Deployed Version?

The core `run_nps_agent` in [`npsagent.py`](./npsagent.py) is identical to the evaluate notebook. The only additions are described below.

### `NPSResponsesAgent` — MLflow Serving Wrapper

MLflow serves models over HTTP via a `POST /invocations` endpoint. To plug our agent into this, we subclass [`ResponsesAgent`](https://mlflow.org/docs/latest/python_api/mlflow.pyfunc.html) — MLflow's standard interface for chat-style models. The `predict` method receives the request, extracts the user message, calls `run_nps_agent`, and returns the result in MLflow's Responses format. This is what turns our agent into a deployable HTTP service.

Authentication to RHOAI's MLflow is handled by `MLFLOW_TRACKING_AUTH=kubernetes` in [`nps-agent.yaml`](./nps-agent.yaml) — no auth code needed in the agent itself. This requires `mlflow>=3.10.0`, so we use the [Red Hat build of MLflow](https://github.com/opendatahub-io/mlflow) from GitHub.

See [`npsagent.py`](./npsagent.py) for the full code.

---

## Prerequisites

Before you begin, make sure you have:

- An OpenShift cluster with RHOAI and MLflow configured
- The `oc` CLI installed and logged in to your cluster
- An [OpenAI API key](https://platform.openai.com/api-keys)
- An [NPS API key](https://www.nps.gov/subjects/developer/get-started.htm) (free and instant)
- A `.env` file in the repository root with your `OPENAI_API_KEY` and `NPS_API_KEY`

In [219]:
import os
import subprocess
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")

## OpenAI Environment Variables
os.environ.setdefault("OPENAI_API_KEY", "")
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
os.environ.setdefault("OPENAI_MODEL_NAME", "gpt-4o-mini")

## NPS API Key
os.environ.setdefault("NPS_API_KEY", "")

# Check that required vars are set
required_vars = ["OPENAI_API_KEY", "OPENAI_BASE_URL", "OPENAI_MODEL_NAME", "NPS_API_KEY"]
if any(not os.getenv(var) for var in required_vars):
    raise ValueError("One or more required environment variables are not set. Check your .env file.")

## Step 1 — Create an OpenShift Project

Each deployment lives in its own OpenShift namespace. Set yours below — use `nps-agent-<yourname>` to avoid conflicts with other users on the same cluster. `MLFLOW_TRACKING_URI` is read from your `.env` file.

In [221]:
NAMESPACE = "nehanth"

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "")
MLFLOW_EXPERIMENT_NAME = "nps-agent"

In [222]:
!oc new-project {NAMESPACE}

Already on project "nehanth" on server "https://api.g4b1z8e6c6k7s6n.2rkm.p3.openshiftapps.com:443".

You can add applications to this project with the 'new-app' command. For example, try:

    oc new-app rails-postgresql-example

to build a new example application in Ruby. Or use kubectl to deploy a simple Kubernetes application:

    kubectl create deployment hello-node --image=registry.k8s.io/e2e-test-images/agnhost:2.43 -- /agnhost serve-hostname



## Step 2 — Create the Secret with API Keys

Store your API keys as an OpenShift Secret so the running pod can access them without baking credentials into the image.

In [223]:
!oc create secret generic nps-agent-secrets \
  --from-literal=OPENAI_API_KEY="{os.getenv('OPENAI_API_KEY')}" \
  --from-literal=NPS_API_KEY="{os.getenv('NPS_API_KEY')}" \
  -n {NAMESPACE}

secret/nps-agent-secrets created


## Step 3 — Apply the OpenShift Template

Process [`nps-agent.yaml`](./nps-agent.yaml) with your cluster values and apply all resources (see [Deployment Files](#Deployment-Files-Overview) above for what each resource does).

In [224]:
!oc process -f ./nps-agent.yaml \
  -p NAMESPACE="{NAMESPACE}" \
  -p MLFLOW_TRACKING_URI="{MLFLOW_TRACKING_URI}" \
  -p MLFLOW_EXPERIMENT_NAME="{MLFLOW_EXPERIMENT_NAME}" \
  | oc apply -f -

buildconfig.build.openshift.io/nps-agent created
imagestream.image.openshift.io/nps-agent created
deployment.apps/nps-agent created
service/nps-agent created
route.route.openshift.io/nps-agent created


## Step 4 — Wait for the s2i Build

The BuildConfig triggers automatically. Watch until you see **"Push successful"**.

In [225]:
!oc logs -f build/nps-agent-1 -n {NAMESPACE}

Cloning "https://github.com/Nehanth/nps_agent.git" ...
	Commit:	283730f21e3b3ba4b58b272a0fd3fd6645921329 (Switch to MLFLOW_TRACKING_AUTH=kubernetes for RHOAI auth)
	Author:	Nehanth <nehanthnarendrula@gmail.com>
	Date:	Mon Feb 16 16:04:29 2026 -0500
time="2026-02-16T21:05:43Z" level=info msg="Not using native diff for overlay, this may cause degraded performance for building images: kernel has CONFIG_OVERLAY_FS_REDIRECT_DIR enabled"
I0216 21:05:43.223668       1 defaults.go:112] Defaulting to storage driver "overlay" with options [mountopt=metacopy=on].
Caching blobs under "/var/cache/blobs".
Trying to pull image-registry.openshift-image-registry.svc:5000/openshift/python@sha256:a1b2f4b9f75ee731a1f05a5d4e23e01d549d9f925325095c285171b30c6f7236...
Getting image source signatures
Copying blob sha256:21c2619d793e92e4d28c8e1ab010e0e57111995e239e8ef0861136b79a913e6e
Copying blob sha256:99bc10fdec926285cd342a154d6e49c76da56f8b572ff75745c77cd74237ddf5
Copying blob sha256:c98636dfe4d3815a89988be

## Step 6 — Verify the Pod is Running

Once the build completes and the image is pushed, the Deployment rolls out a pod. Check that it's in `Running` state before continuing.

In [244]:
!oc get pods -n {NAMESPACE}

NAME                         READY   STATUS        RESTARTS   AGE
nps-agent-1-build            0/1     Completed     0          3m10s
nps-agent-66f85cdfd8-vp6qh   1/1     Running       0          61s
nps-agent-6dd7f6dc8b-twcbs   0/1     Terminating   0          3m10s


## Step 7 — Get the Route URL

The OpenShift Route exposes the agent pod as a public HTTPS endpoint. Grab the hostname so we can call it.

In [245]:
ROUTE_HOST = subprocess.check_output(
    ["oc", "get", "route", "nps-agent", "-n", NAMESPACE, "-o", "jsonpath={.spec.host}"]
).decode().strip()

AGENT_URL = f"https://{ROUTE_HOST}"
print(f"Agent URL: {AGENT_URL}")

Agent URL: https://nps-agent-nehanth.apps.rosa.g4b1z8e6c6k7s6n.2rkm.p3.openshiftapps.com


## Step 8 — Test the Agent

Send a sample question to the deployed agent's `/invocations` endpoint and display the response.

In [246]:
import requests
from IPython.display import display, Markdown

payload = {
    "input": [
        {"role": "user", "content": "What national parks are in California?"}
    ]
}

resp = requests.post(
    f"{AGENT_URL}/invocations",
    headers={"Content-Type": "application/json"},
    json=payload,
    timeout=300,
)

print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    result = resp.json()
    output_text = result.get("output", [{}])[0].get("text", str(result))
    display(Markdown(output_text))
else:
    print(f"Error: {resp.text}")

Status: 200


{'object': 'response', 'output': [{'type': 'message', 'id': 'msg_1', 'content': [{'text': "Here are some national parks and sites in California:\n\n1. **Alcatraz Island**\n   - Description: Explore the island's history of incarceration and its natural beauty.\n   - [More Info](https://www.nps.gov/alca/index.htm)\n\n2. **Butterfield Overland National Historic Trail**\n   - Description: Follow the historic mail route connecting the eastern U.S. to the Far West.\n   - [More Info](https://www.nps.gov/buov/index.htm)\n\n3. **Cabrillo National Monument**\n   - Description: Discover the story of early European exploration and the region's natural resources.\n   - [More Info](https://www.nps.gov/cabr/index.htm)\n\n4. **California National Historic Trail**\n   - Description: Explore the path taken by over 250,000 emigrants to California during the Gold Rush.\n   - [More Info](https://www.nps.gov/cali/index.htm)\n\n5. **Castle Mountains National Monument**\n   - Description: Experience unique desert landscapes, including Joshua tree forests.\n   - [More Info](https://www.nps.gov/camo/index.htm)\n\n6. **Channel Islands National Park**\n   - Description: Explore the unique flora, fauna, and archeological resources of these islands.\n   - [More Info](https://www.nps.gov/chis/index.htm)\n\n7. **César E. Chávez National Monument**\n   - Description: Celebrate the life and legacy of labor leader and civil rights activist César Chávez.\n   - [More Info](https://www.nps.gov/cech/index.htm)\n\n8. **Death Valley National Park**\n   - Description: Discover a land of extremes in this below-sea-level basin.\n   - [More Info](https://www.nps.gov/deva/index.htm)\n\n9. **Devils Postpile National Monument**\n   - Description: View the rare geological formation and the stunning Rainbow Falls.\n   - [More Info](https://www.nps.gov/depo/index.htm)\n\n10. **Eugene O'Neill National Historic Site**\n    - Description: Visit the home of America's only Nobel Prize-winning playwright.\n    - [More Info](https://www.nps.gov/euon/index.htm)\n\nExplore these parks' unique histories and natural wonders!", 'type': 'output_text'}], 'role': 'assistant'}]}

## Step 9 — View Traces in MLflow

Every request is auto-traced. Configure your local MLflow client to connect to the RHOAI server, then use `mlflow.search_traces()` to view them inline.

In [248]:
import mlflow

os.environ["MLFLOW_TRACKING_AUTH"] = "kubernetes"

exp = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
mlflow.search_traces(experiment_ids=[exp.experiment_id], max_results=5, order_by=["timestamp DESC"])

/Users/nnarendr/Documents/Repos/nps_agent/.venv/lib/python3.13/site-packages/mlflow/tracking/request_auth/registry.py:55: UserWarning: Could not find any registered plugin for kubernetes. No authentication header will be added. Please check your provider documentation for installing the right plugin or correct provider name.
  warnings.warn(
/var/folders/xh/zskr16mx55l_0_j2wvxqkpbw0000gn/T/ipykernel_36136/322937242.py:6: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  mlflow.search_traces(experiment_ids=[exp.experiment_id], max_results=5, order_by=["timestamp DESC"])


,trace_id,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-986b92a6e59f5fef8c1fb719e3872277,"{""info"": {""trace_id"": ""tr-986b92a6e59f5fef8c1f...",None,OK,1771276137337,26646,"{'request': {'tool_choice': None, 'truncation'...","{'tool_choice': None, 'truncation': None, 'id'...","{'mlflow.traceInputs': '{""request"": {""tool_cho...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'mGuSpuWfX++MH7cZ44cidw==', 'spa...",[]
1,tr-37a0447c0d01f1d3315ee527c67f91fe,"{""info"": {""trace_id"": ""tr-37a0447c0d01f1d3315e...",None,OK,1771274737488,12610,"{'request': {'tool_choice': None, 'truncation'...","{'tool_choice': None, 'truncation': None, 'id'...","{'mlflow.traceInputs': '{""request"": {""tool_cho...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'N6BEfA0B8dMxXuUnxn+R/g==', 'spa...",[]
2,tr-c2f7abf74814b9b09d3d85148487dafe,"{""info"": {""trace_id"": ""tr-c2f7abf74814b9b09d3d...",None,OK,1771268261819,11200,"{'request': {'tool_choice': None, 'truncation'...","{'tool_choice': None, 'truncation': None, 'id'...","{'mlflow.traceInputs': '{""request"": {""tool_cho...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'wver90gUubCdPYUUhIfa/g==', 'spa...",[]
3,tr-37ab615b2cfe79befc07690519b6e128,"{""info"": {""trace_id"": ""tr-37ab615b2cfe79befc07...",None,OK,1771267547507,34310,"{'request': {'tool_choice': None, 'truncation'...","{'tool_choice': None, 'truncation': None, 'id'...","{'mlflow.traceInputs': '{""request"": {""tool_cho...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'N6thWyz+eb78B2kFGbbhKA==', 'spa...",[]
4,tr-5e44cb3543d90953b0a78993dc48fef4,"{""info"": {""trace_id"": ""tr-5e44cb3543d90953b0a7...",None,OK,1771266823853,11655,"{'request': {'tool_choice': None, 'truncation'...","{'tool_choice': None, 'truncation': None, 'id'...","{'mlflow.traceInputs': '{""request"": {""tool_cho...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'XkTLNUPZCVOwp4mT3Ej+9A==', 'spa...",[]


## Rebuilding After Code Changes

Push changes to the `deploydemo` branch, then trigger a new build:

In [ ]:
!oc start-build nps-agent -n {NAMESPACE}
!oc logs -f build/nps-agent-2 -n {NAMESPACE}

## Cleanup

To delete everything and start from scratch:

In [218]:
!oc delete project {NAMESPACE}

project.project.openshift.io "nehanth" deleted
